In [2]:
# -*- coding: utf-8 -*-
"""
专业层次批量生成脚本（openpyxl 版）

功能：
1. 目标 Excel 中如果已有【专业层次】字段：按规则覆盖旧值。
2. 如果没有【专业层次】字段：自动新增该列。
3. 支持单文件或目录批量处理。
4. 引用两个规则表：
   - 本科层次职业学校名单表：院校名称命中后先按规则2填充【高职本科】
   - 专业层次唯一表：专业名称命中后返回该表中的【专业层次】
5. 最后仅在控制台输出每个文件【学制】字段为空的数据条数，不生成报告文件。

规则优先级：
- 先执行规则2：院校名称命中【本科层次职业学校】名单 -> 高职本科
- 再执行规则1覆盖规则2：
  1) 批次包含“专科” -> 专科
  2) 生源地不是山东/浙江，且批次包含“高本贯通” -> 专科
  3) 生源地是山东/浙江，且学制 <= 3，且专业名称不是“本科预科班” -> 专科
  4) 专业备注包含本科层次职业教育关键词，且学制 >= 4 -> 高职本科
  5) 专业名称命中【专业层次唯一表】 -> 使用该表对应专业层次
  6) 其他仍为空 -> 本科

依赖：
pip install openpyxl
"""

import os
import re
import argparse
from typing import Dict, Optional, Tuple, List
from openpyxl import load_workbook


# =========================
# 路径配置：直接运行时改这里
# =========================

# 目标数据：支持单个 Excel 文件 或 文件夹
INPUT_PATH = r"/Users/bangongyi/Desktop/26年招生计划清洗/内蒙古/mediate_文件/邪"

# 输出目录
OUTPUT_DIR = r"/Users/bangongyi/Desktop/26年招生计划清洗/内蒙古/mediate_文件/魔"

# 本科层次职业学校名单表
# 表头字段需要有：本科层次职业学校
VOCATIONAL_SCHOOL_FILE = r"../专业层次处理/规则表/全国124所职业本科院校（含公示）名单.xlsx"
VOCATIONAL_SCHOOL_SHEET = None
VOCATIONAL_SCHOOL_COL = "本科层次职业学校"

# 专业层次唯一表
# 表头字段需要有：专业名称、专业层次
MAJOR_LEVEL_UNIQUE_FILE = r"../专业层次处理/规则表/Definitely-major.xlsx"
MAJOR_LEVEL_UNIQUE_SHEET = None
MAJOR_LEVEL_UNIQUE_MAJOR_COL = "专业名称"
MAJOR_LEVEL_UNIQUE_LEVEL_COL = "专业层次"

# 如果专业层次唯一表文件不存在：
# True  = 直接报错，避免漏用规则表
# False = 跳过唯一表规则
REQUIRE_MAJOR_LEVEL_UNIQUE_FILE = True

# 如果本科层次职业学校名单表不存在：
# True  = 直接报错
# False = 跳过规则2
REQUIRE_VOCATIONAL_SCHOOL_FILE = True


# =========================
# 字段配置
# =========================

COL_PROVINCE = "生源地"
COL_BATCH = "批次"
COL_SCHOOL = "院校名称"
COL_MAJOR_NAME = "专业名称"
COL_DURATION = "学制"
COL_REMARK = "专业备注"
COL_LEVEL = "专业层次"

# 如果【专业层次】列不存在，默认插入到这个字段后面；找不到则追加到最后
INSERT_AFTER_COL = "最高位次"

OUTPUT_SUFFIX = "_已生成专业层次"

# 专业备注命中这些关键词，并且学制 >= 4，则填充高职本科
VOCATIONAL_BACHELOR_REMARK_KEYWORDS = (
    "本科层次职业教育",
    "职业教育本科",
    "高等职业教育普通本科",
)

# 山东、浙江特殊规则
SPECIAL_PROVINCES_FOR_DURATION = {"山东", "山东省", "浙江", "浙江省"}

# 这些专业名称不参与“山东/浙江 学制<=3 -> 专科”规则
EXCLUDE_MAJOR_FOR_SD_ZJ_SPECIAL = {"本科预科班"}

# 专业层次合法值不强制限制，但这里做一下常见归一
LEVEL_VALUE_NORMALIZE = {
    "高职本科": "高职本科",
    "职业本科": "高职本科",
    "本科层次职业教育": "高职本科",
    "本科": "本科",
    "专科": "专科",
    "高职专科": "专科",
}


# =========================
# 基础工具
# =========================

def ensure_dir(path: str) -> None:
    if path and not os.path.exists(path):
        os.makedirs(path, exist_ok=True)


def get_str(value) -> str:
    return "" if value is None else str(value).strip()


def is_blank(value) -> bool:
    return value is None or str(value).strip() == ""


def norm_text(value) -> str:
    """
    通用文本归一：去空白、统一括号、统一常见符号。
    用于院校名称、专业名称等精确匹配。
    """
    s = get_str(value)
    if not s:
        return ""
    s = s.replace("（", "(").replace("）", ")")
    s = s.replace("　", "").replace("\u3000", "")
    s = re.sub(r"\s+", "", s)
    s = s.replace("·", "").replace("・", "")
    return s


def normalize_level_value(value: str) -> str:
    s = get_str(value)
    if not s:
        return ""
    return LEVEL_VALUE_NORMALIZE.get(s, s)


def list_excel_files(path: str) -> List[str]:
    """
    支持：
    - 单个 .xlsx/.xlsm
    - 文件夹下所有 .xlsx/.xlsm
    """
    if os.path.isfile(path):
        if path.startswith("~$"):
            return []
        if path.lower().endswith((".xlsx", ".xlsm")):
            return [path]
        raise ValueError(f"输入文件不是 .xlsx/.xlsm：{path}")

    if not os.path.isdir(path):
        raise FileNotFoundError(f"输入路径不存在：{path}")

    files = []
    for name in os.listdir(path):
        if name.startswith("~$"):
            continue
        if name.lower().endswith((".xlsx", ".xlsm")):
            files.append(os.path.join(path, name))
    return sorted(files)


def get_sheet(wb, sheet_name_or_none):
    if sheet_name_or_none:
        return wb[sheet_name_or_none]
    return wb.worksheets[0]


def get_header_map(ws, header_row: int = 1) -> Dict[str, int]:
    header_map = {}
    for col in range(1, ws.max_column + 1):
        value = ws.cell(row=header_row, column=col).value
        name = get_str(value)
        if name:
            header_map[name] = col
    return header_map


def ensure_col_after(ws, col_name: str, after_col_name: str, header_row: int = 1) -> Tuple[int, Dict[str, int]]:
    """
    如果列已存在：不移动，只返回列号。
    如果列不存在：插入到 after_col_name 后；如果找不到 after_col_name，则追加到最后。
    """
    header_map = get_header_map(ws, header_row)

    if col_name in header_map:
        return header_map[col_name], header_map

    if after_col_name and after_col_name in header_map:
        insert_at = header_map[after_col_name] + 1
        ws.insert_cols(insert_at, 1)
        ws.cell(row=header_row, column=insert_at).value = col_name
    else:
        insert_at = ws.max_column + 1
        ws.cell(row=header_row, column=insert_at).value = col_name

    header_map = get_header_map(ws, header_row)
    return header_map[col_name], header_map


# =========================
# 学制解析
# =========================

CN_NUM_MAP = {
    "零": 0, "〇": 0,
    "一": 1, "二": 2, "两": 2, "三": 3, "四": 4, "五": 5,
    "六": 6, "七": 7, "八": 8, "九": 9, "十": 10,
}


def chinese_number_to_int(text: str) -> Optional[int]:
    """
    支持：一、二、三、四、十、十一、十二、二十、二十一
    """
    s = get_str(text)
    if not s:
        return None

    m = re.search(r"[零〇一二两三四五六七八九十]+", s)
    if not m:
        return None

    cn = m.group(0)

    if cn == "十":
        return 10

    if "十" in cn:
        left, _, right = cn.partition("十")
        tens = CN_NUM_MAP.get(left, 1) if left else 1
        ones = CN_NUM_MAP.get(right, 0) if right else 0
        return tens * 10 + ones

    return CN_NUM_MAP.get(cn)


def parse_duration(value) -> Optional[int]:
    """
    解析学制字段，返回整数年限。

    支持：
    - 3 / 4 / 3.0 / 4年 / 学制4年
    - 三 / 四 / 三年 / 四年 / 七年医 / 八年制 / 十一年
    """
    if value is None:
        return None

    if isinstance(value, bool):
        return None

    if isinstance(value, int):
        return value

    if isinstance(value, float):
        # 3.0 -> 3；3.5 这种不作为整数学制
        if value.is_integer():
            return int(value)
        return None

    s = get_str(value)
    if not s:
        return None

    # 先识别阿拉伯数字
    m = re.search(r"\d+(?:\.0+)?", s)
    if m:
        num = m.group(0)
        try:
            return int(float(num))
        except Exception:
            return None

    # 再识别中文数字
    return chinese_number_to_int(s)


def duration_le(value, threshold: int) -> bool:
    n = parse_duration(value)
    return n is not None and n <= threshold


def duration_ge(value, threshold: int) -> bool:
    n = parse_duration(value)
    return n is not None and n >= threshold


# =========================
# 规则表读取
# =========================

def load_vocational_school_set(path: str) -> set:
    path = get_str(path)
    if not path or not os.path.exists(path):
        if REQUIRE_VOCATIONAL_SCHOOL_FILE:
            raise FileNotFoundError(f"本科层次职业学校名单表不存在：{path}")
        return set()

    wb = load_workbook(path, data_only=True)
    ws = get_sheet(wb, VOCATIONAL_SCHOOL_SHEET)
    header_map = get_header_map(ws)

    if VOCATIONAL_SCHOOL_COL not in header_map:
        raise ValueError(f"本科层次职业学校名单表缺少字段：{VOCATIONAL_SCHOOL_COL}")

    col = header_map[VOCATIONAL_SCHOOL_COL]
    schools = set()

    for row in range(2, ws.max_row + 1):
        value = ws.cell(row=row, column=col).value
        if is_blank(value):
            continue

        # 支持一个单元格多个院校，用常见分隔符拆开
        for part in re.split(r"[,，;；、\n\r\t]+", str(value)):
            key = norm_text(part)
            if key:
                schools.add(key)

    return schools


def load_major_level_unique_map(path: str) -> Dict[str, str]:
    path = get_str(path)
    if not path or not os.path.exists(path):
        if REQUIRE_MAJOR_LEVEL_UNIQUE_FILE:
            raise FileNotFoundError(f"专业层次唯一表不存在：{path}")
        return {}

    wb = load_workbook(path, data_only=True)
    ws = get_sheet(wb, MAJOR_LEVEL_UNIQUE_SHEET)
    header_map = get_header_map(ws)

    for need in [MAJOR_LEVEL_UNIQUE_MAJOR_COL, MAJOR_LEVEL_UNIQUE_LEVEL_COL]:
        if need not in header_map:
            raise ValueError(f"专业层次唯一表缺少字段：{need}")

    col_major = header_map[MAJOR_LEVEL_UNIQUE_MAJOR_COL]
    col_level = header_map[MAJOR_LEVEL_UNIQUE_LEVEL_COL]

    result = {}

    for row in range(2, ws.max_row + 1):
        major = norm_text(ws.cell(row=row, column=col_major).value)
        level = normalize_level_value(ws.cell(row=row, column=col_level).value)
        if not major or not level:
            continue
        # 如果有重复专业名称，以后出现的同名覆盖前面的
        result[major] = level

    return result


# =========================
# 专业层次判定核心
# =========================

def has_vocational_bachelor_remark(remark: str) -> bool:
    text = get_str(remark)
    if not text:
        return False
    return any(key in text for key in VOCATIONAL_BACHELOR_REMARK_KEYWORDS)


def get_major_level(
    province,
    batch,
    school,
    major_name,
    duration_raw,
    remark,
    vocational_school_set: set,
    major_level_map: Dict[str, str],
) -> str:
    """
    执行顺序：
    1. 先跑规则2：院校名称命中本科层次职业学校名单 -> 高职本科
    2. 再跑规则1覆盖
    3. 最后空值 -> 本科
    """
    province_raw = get_str(province)
    batch_raw = get_str(batch)
    school_key = norm_text(school)
    major_key = norm_text(major_name)
    major_raw = get_str(major_name)

    level = ""

    # =========================
    # 规则2：先填一遍
    # =========================
    if school_key and school_key in vocational_school_set:
        level = "高职本科"

    # =========================
    # 规则1：覆盖规则2
    # =========================

    # 1) 批次包含“专科” -> 专科
    if any(key in batch_raw for key in ("专科", "高职", "高专")):
        level = "专科"

    # 2) 非山东/浙江，批次包含“高本贯通” -> 专科
    if province_raw not in SPECIAL_PROVINCES_FOR_DURATION and "高本贯通" in batch_raw:
        level = "专科"

    # 3) 山东/浙江：学制 <= 3 且专业名称不是“本科预科班” -> 专科
    if province_raw in SPECIAL_PROVINCES_FOR_DURATION:
        if duration_le(duration_raw, 3) and norm_text(major_raw) not in {norm_text(x) for x in EXCLUDE_MAJOR_FOR_SD_ZJ_SPECIAL}:
            level = "专科"

    # 4) 专业备注命中职业本科关键词，且学制 >= 4 -> 高职本科
    if has_vocational_bachelor_remark(remark) and duration_ge(duration_raw, 4):
        level = "高职本科"

    # 5) 专业名称命中专业层次唯一表 -> 使用唯一表对应结果
    # 如果你不希望唯一表覆盖前面规则，可改成：
    # if not level and major_key in major_level_map:
    if major_key and major_key in major_level_map:
        level = major_level_map[major_key]

    # 6) 其他为空 -> 本科
    if not level:
        level = "本科"

    return level


# =========================
# 文件处理
# =========================

def process_one_file(
    filepath: str,
    vocational_school_set: set,
    major_level_map: Dict[str, str],
) -> Tuple[str, int, int]:
    """
    返回：
    - 输出文件路径
    - 学制为空条数
    - 数据总行数
    """
    keep_vba = filepath.lower().endswith(".xlsm")
    wb = load_workbook(filepath, keep_vba=keep_vba)
    ws = wb.worksheets[0]

    col_level, header_map = ensure_col_after(ws, COL_LEVEL, INSERT_AFTER_COL)

    required_cols = [
        COL_PROVINCE,
        COL_BATCH,
        COL_SCHOOL,
        COL_MAJOR_NAME,
        COL_DURATION,
        COL_REMARK,
    ]
    missing = [c for c in required_cols if c not in header_map]
    if missing:
        raise ValueError(f"[{os.path.basename(filepath)}] 缺少字段：{missing}")

    col_province = header_map[COL_PROVINCE]
    col_batch = header_map[COL_BATCH]
    col_school = header_map[COL_SCHOOL]
    col_major = header_map[COL_MAJOR_NAME]
    col_duration = header_map[COL_DURATION]
    col_remark = header_map[COL_REMARK]

    duration_blank_count = 0
    data_rows = max(0, ws.max_row - 1)

    for row in range(2, ws.max_row + 1):
        province = ws.cell(row=row, column=col_province).value
        batch = ws.cell(row=row, column=col_batch).value
        school = ws.cell(row=row, column=col_school).value
        major_name = ws.cell(row=row, column=col_major).value
        duration_raw = ws.cell(row=row, column=col_duration).value
        remark = ws.cell(row=row, column=col_remark).value

        if is_blank(duration_raw):
            duration_blank_count += 1

        level = get_major_level(
            province=province,
            batch=batch,
            school=school,
            major_name=major_name,
            duration_raw=duration_raw,
            remark=remark,
            vocational_school_set=vocational_school_set,
            major_level_map=major_level_map,
        )

        # 覆盖已有专业层次值
        ws.cell(row=row, column=col_level).value = level

    ensure_dir(OUTPUT_DIR)

    base, ext = os.path.splitext(os.path.basename(filepath))
    out_path = os.path.join(OUTPUT_DIR, f"{base}{OUTPUT_SUFFIX}{ext}")

    wb.save(out_path)
    return out_path, duration_blank_count, data_rows


def run(
    input_path: str = INPUT_PATH,
    output_dir: str = OUTPUT_DIR,
    vocational_school_file: str = VOCATIONAL_SCHOOL_FILE,
    major_level_unique_file: str = MAJOR_LEVEL_UNIQUE_FILE,
):
    global OUTPUT_DIR
    OUTPUT_DIR = output_dir

    ensure_dir(OUTPUT_DIR)

    vocational_school_set = load_vocational_school_set(vocational_school_file)
    major_level_map = load_major_level_unique_map(major_level_unique_file)

    print("=" * 100)
    print("专业层次生成开始")
    print(f"输入路径：{input_path}")
    print(f"输出目录：{OUTPUT_DIR}")
    print(f"本科层次职业学校数量：{len(vocational_school_set)}")
    print(f"专业层次唯一表专业数量：{len(major_level_map)}")
    print("=" * 100)

    files = list_excel_files(input_path)
    if not files:
        print("未发现可处理的 .xlsx/.xlsm 文件")
        return []

    summaries = []
    ok = 0
    fail = 0

    for file in files:
        try:
            out_path, duration_blank_count, data_rows = process_one_file(
                file,
                vocational_school_set=vocational_school_set,
                major_level_map=major_level_map,
            )
            summaries.append({
                "file": os.path.basename(file),
                "out_path": out_path,
                "duration_blank_count": duration_blank_count,
                "data_rows": data_rows,
            })
            ok += 1
            print(f"[OK] {os.path.basename(file)} -> {os.path.basename(out_path)} | 学制为空：{duration_blank_count} 条")
        except Exception as e:
            fail += 1
            print(f"[FAIL] {os.path.basename(file)} | {e}")

    print("=" * 100)
    print("处理完成")
    print(f"成功：{ok} 个文件")
    print(f"失败：{fail} 个文件")
    print("-" * 100)
    print("各文件【学制】字段为空统计：")
    for item in summaries:
        print(f"{item['file']}：{item['duration_blank_count']} 条 / 数据行 {item['data_rows']} 条")
    print("=" * 100)

    return summaries


def parse_args():
    parser = argparse.ArgumentParser(description="批量生成/覆盖 Excel 专业层次字段")
    parser.add_argument("--input", "--input-path", dest="input_path", default=INPUT_PATH, help="目标 Excel 文件或目录")
    parser.add_argument("--output-dir", dest="output_dir", default=OUTPUT_DIR, help="输出目录")
    parser.add_argument("--school-file", dest="school_file", default=VOCATIONAL_SCHOOL_FILE, help="本科层次职业学校名单表")
    parser.add_argument("--major-level-file", dest="major_level_file", default=MAJOR_LEVEL_UNIQUE_FILE, help="专业层次唯一表")

    # 兼容 Jupyter / ipykernel：
    # Jupyter 会自动传入类似 --f=/Users/.../kernel-xxx.json 的参数，
    # 普通 parse_args() 会因此报 unrecognized arguments。
    args, _unknown = parser.parse_known_args()
    return args


def main():
    args = parse_args()
    run(
        input_path=args.input_path,
        output_dir=args.output_dir,
        vocational_school_file=args.school_file,
        major_level_unique_file=args.major_level_file,
    )


if __name__ == "__main__":
    main()


专业层次生成开始
输入路径：/Users/bangongyi/Desktop/26年招生计划清洗/内蒙古/mediate_文件/邪
输出目录：/Users/bangongyi/Desktop/26年招生计划清洗/内蒙古/mediate_文件/魔
本科层次职业学校数量：124
专业层次唯一表专业数量：1902


KeyboardInterrupt: 

In [ ]:
# -*- coding: utf-8 -*-
"""
专业层次批量生成脚本（openpyxl 版）

功能：
1. 目标 Excel 中如果已有【专业层次】字段：按规则覆盖旧值。
2. 如果没有【专业层次】字段：自动新增该列。
3. 支持单文件或目录批量处理。
4. 引用两个规则表：
   - 本科层次职业学校名单表：院校名称命中后先按规则2填充【高职本科】
   - 专业层次唯一表：专业名称命中后返回该表中的【专业层次】
5. 最后仅在控制台输出每个文件【学制】字段为空的数据条数，不生成报告文件。

规则优先级：
- 先执行规则2：院校名称命中【本科层次职业学校】名单 -> 高职本科
- 再执行规则1覆盖规则2：
  1) 批次包含“专科” -> 专科
  2) 生源地不是山东/浙江，且批次包含“高本贯通” -> 专科
  3) 生源地是山东/浙江，且学制 <= 3，且专业名称不是“本科预科班” -> 专科
  4) 专业备注包含本科层次职业教育关键词，且学制 >= 4 -> 高职本科
  5) 专业名称命中【专业层次唯一表】 -> 使用该表对应专业层次
  6) 新增强制规则：学制字段解析结果等于 3 -> 专科
  7) 其他仍为空 -> 本科

依赖：
pip install openpyxl
"""

import os
import re
import argparse
from typing import Dict, Optional, Tuple, List
from openpyxl import load_workbook


# =========================
# 路径配置：直接运行时改这里
# =========================

# 目标数据：支持单个 Excel 文件 或 文件夹
INPUT_PATH = r"/Users/bangongyi/Desktop/26年招生计划清洗/内蒙古/上海/C"

# 输出目录
OUTPUT_DIR = r"/Users/bangongyi/Desktop/26年招生计划清洗/内蒙古/上海/end"


# 本科层次职业学校名单表
# 表头字段需要有：本科层次职业学校
VOCATIONAL_SCHOOL_FILE = r"../专业层次处理/规则表/全国124所职业本科院校（含公示）名单.xlsx"
VOCATIONAL_SCHOOL_SHEET = None
VOCATIONAL_SCHOOL_COL = "本科层次职业学校"

# 专业层次唯一表
# 表头字段需要有：专业名称、专业层次
MAJOR_LEVEL_UNIQUE_FILE = r"../专业层次处理/规则表/Definitely-major.xlsx"
MAJOR_LEVEL_UNIQUE_SHEET = None
MAJOR_LEVEL_UNIQUE_MAJOR_COL = "专业名称"
MAJOR_LEVEL_UNIQUE_LEVEL_COL = "专业层次"

# 如果专业层次唯一表文件不存在：
# True  = 直接报错，避免漏用规则表
# False = 跳过唯一表规则
REQUIRE_MAJOR_LEVEL_UNIQUE_FILE = True

# 如果本科层次职业学校名单表不存在：
# True  = 直接报错
# False = 跳过规则2
REQUIRE_VOCATIONAL_SCHOOL_FILE = True


# =========================
# 字段配置
# =========================

COL_PROVINCE = "生源地"
COL_BATCH = "批次"
COL_SCHOOL = "院校名称"
COL_MAJOR_NAME = "专业名称"
COL_DURATION = "学制"
COL_REMARK = "专业备注"
COL_LEVEL = "专业层次"

# 如果【专业层次】列不存在，默认插入到这个字段后面；找不到则追加到最后
INSERT_AFTER_COL = "最高位次"

OUTPUT_SUFFIX = "_已生成专业层次"

# 专业备注命中这些关键词，并且学制 >= 4，则填充高职本科
VOCATIONAL_BACHELOR_REMARK_KEYWORDS = (
    "本科层次职业教育",
    "职业教育本科",
    "高等职业教育普通本科",
)

# 山东、浙江特殊规则
SPECIAL_PROVINCES_FOR_DURATION = {"山东", "山东省", "浙江", "浙江省"}

# 这些专业名称不参与“山东/浙江 学制<=3 -> 专科”规则
EXCLUDE_MAJOR_FOR_SD_ZJ_SPECIAL = {"本科预科班"}

# 专业层次合法值不强制限制，但这里做一下常见归一
LEVEL_VALUE_NORMALIZE = {
    "高职本科": "高职本科",
    "职业本科": "高职本科",
    "本科层次职业教育": "高职本科",
    "本科": "本科",
    "专科": "专科",
    "高职专科": "专科",
}


# =========================
# 基础工具
# =========================

def ensure_dir(path: str) -> None:
    if path and not os.path.exists(path):
        os.makedirs(path, exist_ok=True)


def get_str(value) -> str:
    return "" if value is None else str(value).strip()


def is_blank(value) -> bool:
    return value is None or str(value).strip() == ""


def norm_text(value) -> str:
    """
    通用文本归一：去空白、统一括号、统一常见符号。
    用于院校名称、专业名称等精确匹配。
    """
    s = get_str(value)
    if not s:
        return ""
    s = s.replace("（", "(").replace("）", ")")
    s = s.replace("　", "").replace("\u3000", "")
    s = re.sub(r"\s+", "", s)
    s = s.replace("·", "").replace("・", "")
    return s


def normalize_level_value(value: str) -> str:
    s = get_str(value)
    if not s:
        return ""
    return LEVEL_VALUE_NORMALIZE.get(s, s)


def list_excel_files(path: str) -> List[str]:
    """
    支持：
    - 单个 .xlsx/.xlsm
    - 文件夹下所有 .xlsx/.xlsm
    """
    if os.path.isfile(path):
        if path.startswith("~$"):
            return []
        if path.lower().endswith((".xlsx", ".xlsm")):
            return [path]
        raise ValueError(f"输入文件不是 .xlsx/.xlsm：{path}")

    if not os.path.isdir(path):
        raise FileNotFoundError(f"输入路径不存在：{path}")

    files = []
    for name in os.listdir(path):
        if name.startswith("~$"):
            continue
        if name.lower().endswith((".xlsx", ".xlsm")):
            files.append(os.path.join(path, name))
    return sorted(files)


def get_sheet(wb, sheet_name_or_none):
    if sheet_name_or_none:
        return wb[sheet_name_or_none]
    return wb.worksheets[0]


def get_header_map(ws, header_row: int = 1) -> Dict[str, int]:
    header_map = {}
    for col in range(1, ws.max_column + 1):
        value = ws.cell(row=header_row, column=col).value
        name = get_str(value)
        if name:
            header_map[name] = col
    return header_map


def ensure_col_after(ws, col_name: str, after_col_name: str, header_row: int = 1) -> Tuple[int, Dict[str, int]]:
    """
    如果列已存在：不移动，只返回列号。
    如果列不存在：插入到 after_col_name 后；如果找不到 after_col_name，则追加到最后。
    """
    header_map = get_header_map(ws, header_row)

    if col_name in header_map:
        return header_map[col_name], header_map

    if after_col_name and after_col_name in header_map:
        insert_at = header_map[after_col_name] + 1
        ws.insert_cols(insert_at, 1)
        ws.cell(row=header_row, column=insert_at).value = col_name
    else:
        insert_at = ws.max_column + 1
        ws.cell(row=header_row, column=insert_at).value = col_name

    header_map = get_header_map(ws, header_row)
    return header_map[col_name], header_map


# =========================
# 学制解析
# =========================

CN_NUM_MAP = {
    "零": 0, "〇": 0,
    "一": 1, "二": 2, "两": 2, "三": 3, "四": 4, "五": 5,
    "六": 6, "七": 7, "八": 8, "九": 9, "十": 10,
}


def chinese_number_to_int(text: str) -> Optional[int]:
    """
    支持：一、二、三、四、十、十一、十二、二十、二十一
    """
    s = get_str(text)
    if not s:
        return None

    m = re.search(r"[零〇一二两三四五六七八九十]+", s)
    if not m:
        return None

    cn = m.group(0)

    if cn == "十":
        return 10

    if "十" in cn:
        left, _, right = cn.partition("十")
        tens = CN_NUM_MAP.get(left, 1) if left else 1
        ones = CN_NUM_MAP.get(right, 0) if right else 0
        return tens * 10 + ones

    return CN_NUM_MAP.get(cn)


def parse_duration(value) -> Optional[int]:
    """
    解析学制字段，返回整数年限。

    支持：
    - 3 / 4 / 3.0 / 4年 / 学制4年
    - 三 / 四 / 三年 / 四年 / 七年医 / 八年制 / 十一年
    """
    if value is None:
        return None

    if isinstance(value, bool):
        return None

    if isinstance(value, int):
        return value

    if isinstance(value, float):
        # 3.0 -> 3；3.5 这种不作为整数学制
        if value.is_integer():
            return int(value)
        return None

    s = get_str(value)
    if not s:
        return None

    # 先识别阿拉伯数字
    m = re.search(r"\d+(?:\.0+)?", s)
    if m:
        num = m.group(0)
        try:
            return int(float(num))
        except Exception:
            return None

    # 再识别中文数字
    return chinese_number_to_int(s)


def duration_le(value, threshold: int) -> bool:
    n = parse_duration(value)
    return n is not None and n <= threshold


def duration_ge(value, threshold: int) -> bool:
    n = parse_duration(value)
    return n is not None and n >= threshold


def duration_eq(value, target: int) -> bool:
    """
    判断学制是否等于指定年限。
    支持：3 / 3.0 / 3年 / 三年 等写法。
    """
    n = parse_duration(value)
    return n is not None and n == target


# =========================
# 规则表读取
# =========================

def load_vocational_school_set(path: str) -> set:
    path = get_str(path)
    if not path or not os.path.exists(path):
        if REQUIRE_VOCATIONAL_SCHOOL_FILE:
            raise FileNotFoundError(f"本科层次职业学校名单表不存在：{path}")
        return set()

    wb = load_workbook(path, data_only=True)
    ws = get_sheet(wb, VOCATIONAL_SCHOOL_SHEET)
    header_map = get_header_map(ws)

    if VOCATIONAL_SCHOOL_COL not in header_map:
        raise ValueError(f"本科层次职业学校名单表缺少字段：{VOCATIONAL_SCHOOL_COL}")

    col = header_map[VOCATIONAL_SCHOOL_COL]
    schools = set()

    for row in range(2, ws.max_row + 1):
        value = ws.cell(row=row, column=col).value
        if is_blank(value):
            continue

        # 支持一个单元格多个院校，用常见分隔符拆开
        for part in re.split(r"[,，;；、\n\r\t]+", str(value)):
            key = norm_text(part)
            if key:
                schools.add(key)

    return schools


def load_major_level_unique_map(path: str) -> Dict[str, str]:
    path = get_str(path)
    if not path or not os.path.exists(path):
        if REQUIRE_MAJOR_LEVEL_UNIQUE_FILE:
            raise FileNotFoundError(f"专业层次唯一表不存在：{path}")
        return {}

    wb = load_workbook(path, data_only=True)
    ws = get_sheet(wb, MAJOR_LEVEL_UNIQUE_SHEET)
    header_map = get_header_map(ws)

    for need in [MAJOR_LEVEL_UNIQUE_MAJOR_COL, MAJOR_LEVEL_UNIQUE_LEVEL_COL]:
        if need not in header_map:
            raise ValueError(f"专业层次唯一表缺少字段：{need}")

    col_major = header_map[MAJOR_LEVEL_UNIQUE_MAJOR_COL]
    col_level = header_map[MAJOR_LEVEL_UNIQUE_LEVEL_COL]

    result = {}

    for row in range(2, ws.max_row + 1):
        major = norm_text(ws.cell(row=row, column=col_major).value)
        level = normalize_level_value(ws.cell(row=row, column=col_level).value)
        if not major or not level:
            continue
        # 如果有重复专业名称，以后出现的同名覆盖前面的
        result[major] = level

    return result


# =========================
# 专业层次判定核心
# =========================

def has_vocational_bachelor_remark(remark: str) -> bool:
    text = get_str(remark)
    if not text:
        return False
    return any(key in text for key in VOCATIONAL_BACHELOR_REMARK_KEYWORDS)


def get_major_level(
    province,
    batch,
    school,
    major_name,
    duration_raw,
    remark,
    vocational_school_set: set,
    major_level_map: Dict[str, str],
) -> str:
    """
    执行顺序：
    1. 先跑规则2：院校名称命中本科层次职业学校名单 -> 高职本科
    2. 再跑规则1覆盖
    3. 强制规则：学制等于 3 -> 专科
    4. 最后空值 -> 本科
    """
    province_raw = get_str(province)
    batch_raw = get_str(batch)
    school_key = norm_text(school)
    major_key = norm_text(major_name)
    major_raw = get_str(major_name)

    level = ""

    # =========================
    # 规则2：先填一遍
    # =========================
    if school_key and school_key in vocational_school_set:
        level = "高职本科"

    # =========================
    # 规则1：覆盖规则2
    # =========================

    # 1) 批次包含“专科” -> 专科
    if any(key in batch_raw for key in ("专科", "高职", "高专")):
        level = "专科"

    # 2) 非山东/浙江，批次包含“高本贯通” -> 专科
    if province_raw not in SPECIAL_PROVINCES_FOR_DURATION and "高本贯通" in batch_raw:
        level = "专科"

    # 3) 山东/浙江：学制 <= 3 且专业名称不是“本科预科班” -> 专科
    if province_raw in SPECIAL_PROVINCES_FOR_DURATION:
        if duration_le(duration_raw, 3) and norm_text(major_raw) not in {norm_text(x) for x in EXCLUDE_MAJOR_FOR_SD_ZJ_SPECIAL}:
            level = "专科"

    # 4) 专业备注命中职业本科关键词，且学制 >= 4 -> 高职本科
    if has_vocational_bachelor_remark(remark) and duration_ge(duration_raw, 4):
        level = "高职本科"

    # 5) 专业名称命中专业层次唯一表 -> 使用唯一表对应结果
    # 如果你不希望唯一表覆盖前面规则，可改成：
    # if not level and major_key in major_level_map:
    if major_key and major_key in major_level_map:
        level = major_level_map[major_key]

    # 6) 新增强制规则：只要学制字段解析结果等于 3，对应专业层次统一填充“专科”。
    # 放在唯一表之后，确保“学制=3 -> 专科”拥有最终覆盖权。
    if duration_eq(duration_raw, 3):
        level = "专科"

    # 7) 其他为空 -> 本科
    if not level:
        level = "本科"

    return level


# =========================
# 文件处理
# =========================

def process_one_file(
    filepath: str,
    vocational_school_set: set,
    major_level_map: Dict[str, str],
) -> Tuple[str, int, int]:
    """
    返回：
    - 输出文件路径
    - 学制为空条数
    - 数据总行数
    """
    keep_vba = filepath.lower().endswith(".xlsm")
    wb = load_workbook(filepath, keep_vba=keep_vba)
    ws = wb.worksheets[0]

    col_level, header_map = ensure_col_after(ws, COL_LEVEL, INSERT_AFTER_COL)

    required_cols = [
        COL_PROVINCE,
        COL_BATCH,
        COL_SCHOOL,
        COL_MAJOR_NAME,
        COL_DURATION,
        COL_REMARK,
    ]
    missing = [c for c in required_cols if c not in header_map]
    if missing:
        raise ValueError(f"[{os.path.basename(filepath)}] 缺少字段：{missing}")

    col_province = header_map[COL_PROVINCE]
    col_batch = header_map[COL_BATCH]
    col_school = header_map[COL_SCHOOL]
    col_major = header_map[COL_MAJOR_NAME]
    col_duration = header_map[COL_DURATION]
    col_remark = header_map[COL_REMARK]

    duration_blank_count = 0
    data_rows = max(0, ws.max_row - 1)

    for row in range(2, ws.max_row + 1):
        province = ws.cell(row=row, column=col_province).value
        batch = ws.cell(row=row, column=col_batch).value
        school = ws.cell(row=row, column=col_school).value
        major_name = ws.cell(row=row, column=col_major).value
        duration_raw = ws.cell(row=row, column=col_duration).value
        remark = ws.cell(row=row, column=col_remark).value

        if is_blank(duration_raw):
            duration_blank_count += 1

        level = get_major_level(
            province=province,
            batch=batch,
            school=school,
            major_name=major_name,
            duration_raw=duration_raw,
            remark=remark,
            vocational_school_set=vocational_school_set,
            major_level_map=major_level_map,
        )

        # 覆盖已有专业层次值
        ws.cell(row=row, column=col_level).value = level

    ensure_dir(OUTPUT_DIR)

    base, ext = os.path.splitext(os.path.basename(filepath))
    out_path = os.path.join(OUTPUT_DIR, f"{base}{OUTPUT_SUFFIX}{ext}")

    wb.save(out_path)
    return out_path, duration_blank_count, data_rows


def run(
    input_path: str = INPUT_PATH,
    output_dir: str = OUTPUT_DIR,
    vocational_school_file: str = VOCATIONAL_SCHOOL_FILE,
    major_level_unique_file: str = MAJOR_LEVEL_UNIQUE_FILE,
):
    global OUTPUT_DIR
    OUTPUT_DIR = output_dir

    ensure_dir(OUTPUT_DIR)

    vocational_school_set = load_vocational_school_set(vocational_school_file)
    major_level_map = load_major_level_unique_map(major_level_unique_file)

    print("=" * 100)
    print("专业层次生成开始")
    print(f"输入路径：{input_path}")
    print(f"输出目录：{OUTPUT_DIR}")
    print(f"本科层次职业学校数量：{len(vocational_school_set)}")
    print(f"专业层次唯一表专业数量：{len(major_level_map)}")
    print("=" * 100)

    files = list_excel_files(input_path)
    if not files:
        print("未发现可处理的 .xlsx/.xlsm 文件")
        return []

    summaries = []
    ok = 0
    fail = 0

    for file in files:
        try:
            out_path, duration_blank_count, data_rows = process_one_file(
                file,
                vocational_school_set=vocational_school_set,
                major_level_map=major_level_map,
            )
            summaries.append({
                "file": os.path.basename(file),
                "out_path": out_path,
                "duration_blank_count": duration_blank_count,
                "data_rows": data_rows,
            })
            ok += 1
            print(f"[OK] {os.path.basename(file)} -> {os.path.basename(out_path)} | 学制为空：{duration_blank_count} 条")
        except Exception as e:
            fail += 1
            print(f"[FAIL] {os.path.basename(file)} | {e}")

    print("=" * 100)
    print("处理完成")
    print(f"成功：{ok} 个文件")
    print(f"失败：{fail} 个文件")
    print("-" * 100)
    print("各文件【学制】字段为空统计：")
    for item in summaries:
        print(f"{item['file']}：{item['duration_blank_count']} 条 / 数据行 {item['data_rows']} 条")
    print("=" * 100)

    return summaries


def parse_args():
    parser = argparse.ArgumentParser(description="批量生成/覆盖 Excel 专业层次字段")
    parser.add_argument("--input", "--input-path", dest="input_path", default=INPUT_PATH, help="目标 Excel 文件或目录")
    parser.add_argument("--output-dir", dest="output_dir", default=OUTPUT_DIR, help="输出目录")
    parser.add_argument("--school-file", dest="school_file", default=VOCATIONAL_SCHOOL_FILE, help="本科层次职业学校名单表")
    parser.add_argument("--major-level-file", dest="major_level_file", default=MAJOR_LEVEL_UNIQUE_FILE, help="专业层次唯一表")

    # 兼容 Jupyter / ipykernel：
    # Jupyter 会自动传入类似 --f=/Users/.../kernel-xxx.json 的参数，
    # 普通 parse_args() 会因此报 unrecognized arguments。
    args, _unknown = parser.parse_known_args()
    return args


def main():
    args = parse_args()
    run(
        input_path=args.input_path,
        output_dir=args.output_dir,
        vocational_school_file=args.school_file,
        major_level_unique_file=args.major_level_file,
    )


if __name__ == "__main__":
    main()


专业层次生成开始
输入路径：/Users/bangongyi/Desktop/26年招生计划清洗/内蒙古/mediate_文件/邪
输出目录：/Users/bangongyi/Desktop/26年招生计划清洗/内蒙古/mediate_文件/魔
本科层次职业学校数量：124
专业层次唯一表专业数量：1902
[OK] 内蒙古26年上传文件-x.xlsx -> 内蒙古26年上传文件-x_已生成专业层次.xlsx | 学制为空：0 条
处理完成
成功：1 个文件
失败：0 个文件
----------------------------------------------------------------------------------------------------
各文件【学制】字段为空统计：
内蒙古26年上传文件-x.xlsx：0 条 / 数据行 24453 条
